# Train custom wake word: **bayan** (Colab-ready)

Clean notebook with all Python 3.13 / PyTorch 2.6 fixes applied.

## Do this
1. **Runtime → Disconnect and delete runtime**
2. **Runtime → Change runtime type → GPU (T4)**
3. Upload this file if needed, then **Runtime → Run all**
4. Wait ~75–90 minutes
5. Download **`bayan.onnx`**
6. Copy `bayan.onnx` into your BayanAi project root
7. In `config.json`: `"wake_word_model": "bayan"`

## Phrases trained
- `bayan`
- `hey bayan`
- `hi bayan`


## 1. Install dependencies

GPU required. Skips piper-phonemize on Python 3.13.


In [ ]:
# ── GPU check FIRST ──────────────────────────────────────────────────
import subprocess as _sp, sys
if _sp.run(['nvidia-smi'], capture_output=True).returncode != 0:
    raise SystemExit(
        '\n' + '=' * 68 +
        '\nNO GPU — stopping.\n'
        'Runtime > Change runtime type > T4 GPU, then Run all again.\n'
        + '=' * 68)
print('  GPU present.')

!apt-get install -y -qq cmake espeak-ng espeak-ng-data libespeak-ng-dev libsndfile1 pkg-config build-essential ffmpeg unzip 2>&1 | tail -3

!pip install -q \
    webrtcvad mutagen==1.47.0 torchinfo torchmetrics pyyaml tqdm datasets \
    soundfile audiomentations torch_audiomentations pronouncing onnxruntime onnx \
    speechbrain acoustics scipy requests huggingface_hub deep-phonemizer

!pip install -q --no-deps piper-tts

if sys.version_info < (3, 13):
    _sp.call([sys.executable, '-m', 'pip', 'install', '-q', 'piper-phonemize-cross'])
else:
    print('  Python 3.13+: skipping piper-phonemize-cross')

print('Python:', sys.version)
import torch
print(f'  torch: {torch.__version__}  cuda: {torch.cuda.is_available()}')
import torchaudio, yaml, soundfile, audiomentations, torch_audiomentations, speechbrain, acoustics, dp
import onnx, onnxruntime, requests
from piper import PiperVoice, SynthesisConfig
# Global torch 2.6+ fix for this runtime
if not getattr(torch.load, '_bayan_patched', False):
    _orig = torch.load
    def _load(*a, **k):
        k.setdefault('weights_only', False)
        return _orig(*a, **k)
    _load._bayan_patched = True
    torch.load = _load
print('  ✓ core imports OK + torch.load patched')


## 2. Clone repos + write bayan helpers + download Piper model


In [ ]:
import os, sys, subprocess
os.chdir('/content')

# Write helpers file
open('/content/bayan_helpers.py', 'w', encoding='utf-8').write('"""Shared Colab helper — also embedded into train_wakeword.ipynb cell 2."""\nimport os\nimport yaml\n\nMODEL_NAME = "bayan"\nTARGET_PHRASE = ["bayan", "hey bayan", "hi bayan"]\n\n\ndef ensure_bayan_config(path="/content/my_model.yaml", force=False):\n    """Create or load my_model.yaml for the bayan wake word."""\n    cfg = {\n        "target_phrase": list(TARGET_PHRASE),\n        "model_name": MODEL_NAME,\n        "custom_negative_phrases": [],\n        "n_samples": 2000,\n        "n_samples_val": 1000,\n        "tts_batch_size": 50,\n        "piper_sample_generator_path": "/content/piper-sample-generator",\n        "augmentation_rounds": 1,\n        "augmentation_batch_size": 16,\n        "steps": 20000,\n        "max_negative_weight": 1500,\n        "target_accuracy": 0.7,\n        "target_recall": 0.5,\n        "target_false_positives_per_hour": 0.5,\n        "batch_size": 128,\n        "learning_rate": 1e-4,\n        "model_type": "dnn",\n        "layer_dim": 128,\n        "layer_size": 128,\n        "n_blocks": 1,\n        "model_input_shape": [16, 96],\n        "n_classes": 1,\n        "batch_n_per_class": {\n            "ACAV100M_sample": 1024,\n            "adversarial_negative": 50,\n            "positive": 50,\n        },\n        "background_paths": ["/content/fma_wav"],\n        "background_paths_duplication_rate": [1],\n        "rir_paths": ["/content/mit_rirs"],\n        "false_positive_validation_data_path": "/content/acav_val_subset.npy",\n        "feature_data_files": {"ACAV100M_sample": "/content/acav_train_subset.npy"},\n        "output_dir": f"/content/{MODEL_NAME}_output",\n        "tflite_export": False,\n        "onnx_export": True,\n        "positive_clips_train_dir": f"/content/{MODEL_NAME}_output/{MODEL_NAME}/positive_train",\n        "positive_clips_test_dir": f"/content/{MODEL_NAME}_output/{MODEL_NAME}/positive_test",\n        "negative_clips_train_dir": f"/content/{MODEL_NAME}_output/{MODEL_NAME}/negative_train",\n        "negative_clips_test_dir": f"/content/{MODEL_NAME}_output/{MODEL_NAME}/negative_test",\n        "feature_save_dir": f"/content/{MODEL_NAME}_output/{MODEL_NAME}",\n    }\n    if force or not os.path.exists(path):\n        os.makedirs(cfg["output_dir"], exist_ok=True)\n        with open(path, "w", encoding="utf-8") as f:\n            yaml.dump(cfg, f, sort_keys=False)\n        print(f"  wrote {path}")\n        return cfg\n    with open(path, encoding="utf-8") as f:\n        loaded = yaml.safe_load(f)\n    return loaded or cfg\n')
print('  ✓ wrote /content/bayan_helpers.py')

PSG_DIR = '/content/piper-sample-generator'
PSG_PIN = '1a8c49bd29b3a132721086ee88f2253f788594a8^'
PSG_GS  = f'{PSG_DIR}/generate_samples.py'
if not os.path.exists(PSG_GS):
    print('  cloning piper-sample-generator...')
    !rm -rf {PSG_DIR}
    !git clone -q https://github.com/rhasspy/piper-sample-generator {PSG_DIR}
!cd {PSG_DIR} && git fetch -q --all && git checkout -q {PSG_PIN}
assert os.path.exists(PSG_GS)
print('  ✓ piper-sample-generator')

PIPER_MODEL = f'{PSG_DIR}/models/en_US-libritts_r-medium.pt'
URL = 'https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt'
if not os.path.exists(PIPER_MODEL) or os.path.getsize(PIPER_MODEL) < 100_000_000:
    !mkdir -p {PSG_DIR}/models
    !wget -q --tries=5 --timeout=300 -O {PIPER_MODEL} {URL}
assert os.path.getsize(PIPER_MODEL) > 100_000_000
print(f'  ✓ libritts {os.path.getsize(PIPER_MODEL)/1e6:.0f} MB')

OWW_DIR = '/content/openwakeword'
if not os.path.exists(f'{OWW_DIR}/openwakeword/train.py'):
    !rm -rf {OWW_DIR}
    !git clone -q https://github.com/dscripka/openwakeword {OWW_DIR}
subprocess.call([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '-e', OWW_DIR])
if OWW_DIR not in sys.path:
    sys.path.insert(0, OWW_DIR)
os.environ['PYTHONPATH'] = OWW_DIR + (os.pathsep + os.environ['PYTHONPATH'] if os.environ.get('PYTHONPATH') else '')
for _m in list(sys.modules):
    if _m.startswith('openwakeword'):
        del sys.modules[_m]
import openwakeword, torch
assert openwakeword.__file__
if not getattr(torch.load, '_bayan_patched', False):
    _orig = torch.load
    def _load(*a, **k):
        k.setdefault('weights_only', False)
        return _orig(*a, **k)
    _load._bayan_patched = True
    torch.load = _load
print('  ✓ openwakeword', openwakeword.__file__)


## 3. Runtime patches


In [ ]:
import os, glob

# Patch A: torch_audiomentations 0.11 calls torchaudio.set_audio_backend
# which torchaudio 2.x removed. sed-replace the bad line with no-op.
for path in glob.glob('/usr/local/lib/python*/dist-packages/torch_audiomentations/utils/io.py'):
    !sed -i 's|torchaudio.set_audio_backend("soundfile")|pass  # patched|' "{path}"
    print(f'  Patch A (set_audio_backend): {path}')

# Patch B: copy generate_samples.py into openwakeword's package dir.
src = '/content/piper-sample-generator/generate_samples.py'
dst = '/content/openwakeword/openwakeword/generate_samples.py'
assert os.path.exists(src), 'piper-sample-generator/generate_samples.py missing — cell 2 failed'
if (not os.path.exists(dst)) or os.path.getsize(dst) != os.path.getsize(src):
    !cp "{src}" "{dst}"
print(f'  Patch B (generate_samples copy): {dst}')

# Patch C: HF Hub default 10s timeouts → 120s
os.environ['HF_HUB_ETAG_TIMEOUT'] = '120'
os.environ['HF_HUB_DOWNLOAD_TIMEOUT'] = '120'
import huggingface_hub.constants as hfc
for attr in ['DEFAULT_ETAG_TIMEOUT', 'DEFAULT_DOWNLOAD_TIMEOUT',
             'HF_HUB_ETAG_TIMEOUT', 'HF_HUB_DOWNLOAD_TIMEOUT']:
    if hasattr(hfc, attr): setattr(hfc, attr, 120)
print('  Patch C (HF Hub timeouts → 120s)')

# Patch D: torchaudio.info shim (removed in torchaudio 2.x)
import torchaudio
init_path = torchaudio.__file__
SHIM_MARKER = '# ── PATCH: info() shim for torchaudio 2.x ──'
with open(init_path) as f:
    content = f.read()
if SHIM_MARKER not in content:
    shim = (f'\n\n{SHIM_MARKER}\n'
            'def info(file_path, *args, **kwargs):\n'
            '    import soundfile as _sf\n'
            '    si = _sf.info(str(file_path))\n'
            '    return type("_TorchaudioInfo", (), {\n'
            '        "num_frames": si.frames, "sample_rate": si.samplerate,\n'
            '        "num_channels": si.channels, "bits_per_sample": 16,\n'
            '        "encoding": "PCM_S",\n'
            '    })()\n')
    with open(init_path, 'a') as f: f.write(shim)
    import importlib; importlib.reload(torchaudio)
print(f'  Patch D (torchaudio.info shim): {hasattr(torchaudio, "info")}')

# Patch E: openwakeword's generate_samples requires `model` arg but train.py
# doesn't pass it — default to the libritts model we already downloaded.
TARGET = '/content/piper-sample-generator/generate_samples.py'
!sed -i 's|model: Union\[str, Path\],|model: Union[str, Path] = "/content/piper-sample-generator/models/en_US-libritts_r-medium.pt",|' "{TARGET}"
!cp "{TARGET}" /content/openwakeword/openwakeword/generate_samples.py
print('  Patch E (generate_samples model arg default)')

# Patch F: train.py val dtype cast (line ~519 + ~547)
TRAIN_PY = '/content/openwakeword/openwakeword/train.py'
!sed -i 's|val_predictions = self.model(x_val)$|val_predictions = self.model(x_val.float())|' "{TRAIN_PY}"
print('  Patch F (train.py val dtype cast)')

# Patch G: DeepPhonemizer's checkpoint pickles a Preprocessor, and torch 2.6+
# defaults torch.load to weights_only=True, which refuses it:
#   UnpicklingError: Unsupported global: GLOBAL dp.preprocessing.text.Preprocessor
# Rewrite dp's own loader. Done in Python (not sed) so it cannot silently
# no-op on a whitespace change, and so we can assert the result.
# weights_only=False is safe here: the checkpoint is fetched by dp itself from
# its own S3 bucket, which this notebook already trusts.
import dp.model.model as _dpm
_DP_PATH = _dpm.__file__
_before = open(_DP_PATH).read()
if 'weights_only=False' not in _before:
    _after = _before.replace(
        'torch.load(checkpoint_path, map_location=device)',
        'torch.load(checkpoint_path, map_location=device, weights_only=False)')
    assert _after != _before, f'Patch G: torch.load call not found in {_DP_PATH}'
    open(_DP_PATH, 'w').write(_after)
assert 'weights_only=False' in open(_DP_PATH).read(), 'Patch G did not stick'
print(f'  Patch G (dp torch.load weights_only): {_DP_PATH}')

# Patch H: torch.load in generate_samples for PyTorch 2.6+
from pathlib import Path as _P
for _path in [
    '/content/piper-sample-generator/generate_samples.py',
    '/content/openwakeword/openwakeword/generate_samples.py',
]:
    _p = _P(_path)
    if not _p.exists():
        continue
    _t = _p.read_text()
    _t2 = _t.replace('torch.load(model_path)', 'torch.load(model_path, map_location="cpu", weights_only=False)')
    if _t2 != _t:
        _p.write_text(_t2)
        print('  Patch H:', _path)
!cp /content/piper-sample-generator/generate_samples.py /content/openwakeword/openwakeword/generate_samples.py
print('  Patch H synced generate_samples into openwakeword package')


## 4. Pre-flight


In [ ]:
import os, importlib, sys, torch
OWW_DIR = '/content/openwakeword'
if OWW_DIR not in sys.path:
    sys.path.insert(0, OWW_DIR)
missing = []
for name, p in {
    'generate_samples': '/content/piper-sample-generator/generate_samples.py',
    'libritts': '/content/piper-sample-generator/models/en_US-libritts_r-medium.pt',
    'oww train': '/content/openwakeword/openwakeword/train.py',
    'helpers': '/content/bayan_helpers.py',
}.items():
    ok = os.path.exists(p)
    print(('✓' if ok else '✗'), name)
    if not ok: missing.append(name)
for mod in ['torch', 'torchaudio', 'piper', 'openwakeword', 'dp', 'yaml', 'soundfile']:
    try: importlib.import_module(mod)
    except Exception as e:
        missing.append(mod); print('✗', mod, e)
if not torch.cuda.is_available():
    raise RuntimeError('NO GPU')
print('CUDA', torch.cuda.get_device_name(0))
if missing:
    raise RuntimeError('Pre-flight failed: ' + str(missing))
print('Pre-flight PASSED')


## 5. Download openwakeword shared models (mel + embedding)

Required by augment_clips' feature-extraction step. Idempotent — skips files already cached.

In [ ]:
import os
models_dir = '/content/openwakeword/openwakeword/resources/models'
os.makedirs(models_dir, exist_ok=True)
BASE = 'https://github.com/dscripka/openWakeWord/releases/download/v0.5.1'
for fname in ['embedding_model.onnx', 'embedding_model.tflite',
              'melspectrogram.onnx', 'melspectrogram.tflite']:
    out = f'{models_dir}/{fname}'
    if os.path.exists(out) and os.path.getsize(out) > 1000:
        print(f'  cached: {fname} ({os.path.getsize(out)/1e3:.0f} KB)')
    else:
        !wget -q --tries=5 --timeout=120 "{BASE}/{fname}" -O "{out}"
        print(f'  ✓ downloaded: {fname} ({os.path.getsize(out)/1e3:.0f} KB)')

## 6. MIT impulse responses → 16-kHz WAVs (~1 min)

Caches the HF dataset, then converts to WAV. Both halves idempotent.

In [ ]:
import os, time, numpy as np
import scipy.io.wavfile as wavfile
from huggingface_hub import snapshot_download
import datasets

output_dir = '/content/mit_rirs'
os.makedirs(output_dir, exist_ok=True)
if len([f for f in os.listdir(output_dir) if f.endswith('.wav')]) >= 250:
    print(f'  cached: {len(os.listdir(output_dir))} MIT IR WAVs')
else:
    # Pre-cache via snapshot_download (more reliable than load_dataset's internal
    # 10s timeout on 270 file metadata calls).
    for i in range(6):
        try:
            snapshot_download(repo_id='davidscripka/MIT_environmental_impulse_responses',
                              repo_type='dataset', etag_timeout=120, max_workers=4)
            break
        except Exception as e:
            wait = 15 * (i+1)
            print(f'  snapshot_download attempt {i+1}/6: {type(e).__name__}: {str(e)[:80]} — sleep {wait}s')
            time.sleep(wait)
    # Load + convert
    for attempt in range(4):
        try:
            rir_dataset = datasets.load_dataset('davidscripka/MIT_environmental_impulse_responses',
                                                split='train', streaming=False)
            break
        except Exception as e:
            print(f'  load_dataset attempt {attempt+1}: {type(e).__name__}: {str(e)[:120]}')
            time.sleep(20 * (attempt + 1))
    else:
        raise RuntimeError('load_dataset failed for MIT IRs')
    n = 0
    for i, row in enumerate(rir_dataset):
        out = f'{output_dir}/{i:04d}.wav'
        if os.path.exists(out): continue
        audio = row['audio']
        sr = audio['sampling_rate']
        arr = np.asarray(audio['array'])
        if sr != 16000:
            from scipy.signal import resample_poly
            arr = resample_poly(arr, 16000, sr)
        arr = (arr * 32767).clip(-32768, 32767).astype(np.int16)
        wavfile.write(out, 16000, arr); n += 1
    print(f'  ✓ wrote {n} new WAVs (total {len(os.listdir(output_dir))})')

## 7. Download FMA + ACAV features (~10-15 min)

FMA = ~8 GB zip of MP3s for background-noise mixing. ACAV = ~17 GB .npy of pre-computed features (becomes negative training corpus). Both use stream + resume so a session restart picks up where it left off.

In [ ]:
import os, requests, zipfile
from tqdm.auto import tqdm
os.chdir('/content')

# ── ACAV features (17 GB) ─────────────────────────────────────────────
ACAV_FULL = '/content/openwakeword_features_ACAV100M_2000_hrs_16bit.npy'
ACAV_URL = 'https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/openwakeword_features_ACAV100M_2000_hrs_16bit.npy'
if os.path.exists(ACAV_FULL) and os.path.getsize(ACAV_FULL) > 16_000_000_000:
    print(f'  cached: ACAV {os.path.getsize(ACAV_FULL)/1e9:.1f} GB')
else:
    resume_byte = os.path.getsize(ACAV_FULL) if os.path.exists(ACAV_FULL) else 0
    headers = {'Range': f'bytes={resume_byte}-'} if resume_byte else {}
    mode = 'ab' if resume_byte else 'wb'
    print(f'  ACAV: {"resuming from" if resume_byte else "fetching"} {resume_byte/1e9:.1f} GB')
    with requests.get(ACAV_URL, stream=True, headers=headers, timeout=120) as r:
        r.raise_for_status()
        total = int(r.headers.get('content-length', 0)) + resume_byte
        with open(ACAV_FULL, mode) as f, tqdm(total=total, initial=resume_byte,
                                              unit='B', unit_scale=True, unit_divisor=1024,
                                              desc='ACAV') as pbar:
            for chunk in r.iter_content(chunk_size=4*1024*1024):
                if chunk: f.write(chunk); pbar.update(len(chunk))
    print(f'  ✓ ACAV: {os.path.getsize(ACAV_FULL)/1e9:.1f} GB')

# ── FMA small (~8 GB zip) with resume + extract ───────────────────────
FMA_DIR = '/content/fma'
FMA_ZIP = '/content/fma_small.zip'
os.makedirs(FMA_DIR, exist_ok=True)
if os.path.isdir(FMA_DIR) and len(os.listdir(FMA_DIR)) > 100:
    print(f'  cached: FMA {len(os.listdir(FMA_DIR))} entries')
else:
    fma_url = 'https://os.unil.cloud.switch.ch/fma/fma_small.zip'
    resume_byte = os.path.getsize(FMA_ZIP) if os.path.exists(FMA_ZIP) else 0
    headers = {'Range': f'bytes={resume_byte}-'} if resume_byte else {}
    mode = 'ab' if resume_byte else 'wb'
    print(f'  FMA: {"resuming from" if resume_byte else "fetching"} {resume_byte/1e9:.1f} GB')
    with requests.get(fma_url, stream=True, headers=headers, timeout=120) as r:
        r.raise_for_status()
        total = int(r.headers.get('content-length', 0)) + resume_byte
        with open(FMA_ZIP, mode) as f, tqdm(total=total, initial=resume_byte,
                                            unit='B', unit_scale=True, unit_divisor=1024,
                                            desc='FMA') as pbar:
            for chunk in r.iter_content(chunk_size=4*1024*1024):
                if chunk: f.write(chunk); pbar.update(len(chunk))
    print(f'  extracting FMA...')
    with zipfile.ZipFile(FMA_ZIP) as zf:
        for name in tqdm(zf.namelist(), desc='extract', unit='f'):
            zf.extract(name, FMA_DIR)
    os.remove(FMA_ZIP)
    print(f'  ✓ FMA extracted: {len(os.listdir(FMA_DIR))} entries')

## 8. FMA MP3s → 16-kHz mono WAVs (~5 min)

audiomentations needs WAV. 1500 conversions = plenty of background-noise variety.

In [ ]:
import os, glob, subprocess
from tqdm.auto import tqdm
out_dir = '/content/fma_wav'
os.makedirs(out_dir, exist_ok=True)
n_existing = len(os.listdir(out_dir))
if n_existing >= 1500:
    print(f'  cached: {n_existing} FMA WAVs')
else:
    mp3s = glob.glob('/content/fma/**/*.mp3', recursive=True)
    if len(mp3s) < 100:
        raise RuntimeError(f'FMA download incomplete — only {len(mp3s)} MP3s found')
    existing = set(os.listdir(out_dir))
    n_target = min(1500, len(mp3s))
    to_convert = [(i, mp3s[i]) for i in range(n_target) if f'{i:05d}.wav' not in existing]
    print(f'  Converting {len(to_convert)} MP3s → 16kHz mono WAVs (skipping {n_target - len(to_convert)} cached)')
    for i, mp3 in tqdm(to_convert):
        out = f'{out_dir}/{i:05d}.wav'
        subprocess.run(['ffmpeg', '-y', '-loglevel', 'error', '-i', mp3,
                        '-ac', '1', '-ar', '16000', out], check=False)
    print(f'  ✓ {len(os.listdir(out_dir))} FMA WAVs ready')

## 9. Subsample ACAV — 1.7 GB train + 170 MB val

Full 17 GB would OOM training pass. **Validation slice stays as raw `(M, 96)`** — the hand-rolled trainer slides a 16-frame window over it itself.

In [ ]:
import numpy as np, os
SRC = '/content/openwakeword_features_ACAV100M_2000_hrs_16bit.npy'
TRAIN_DST = '/content/acav_train_subset.npy'
VAL_DST   = '/content/acav_val_subset.npy'
if os.path.exists(TRAIN_DST) and os.path.exists(VAL_DST):
    train_arr = np.load(TRAIN_DST, mmap_mode='r')
    val_arr = np.load(VAL_DST, mmap_mode='r')
    print(f'  cached: train={train_arr.shape} ({train_arr.nbytes/1e9:.2f} GB), val={val_arr.shape} ({val_arr.nbytes/1e6:.0f} MB)')
else:
    if not os.path.exists(SRC):
        raise RuntimeError(f'ACAV source missing: {SRC} — re-run cell 7')
    arr = np.load(SRC, mmap_mode='r')
    print(f'  source: {arr.shape}, dtype={arr.dtype}')
    n_total = len(arr)
    n_train = n_total // 10
    n_val_rows = n_total // 100
    print(f'  slicing: train=[:{n_train}] (3-D), val=[{n_train}:{n_train+n_val_rows}] (flattened to 2-D)')
    train_chunk = np.array(arr[:n_train])
    np.save(TRAIN_DST, train_chunk)
    print(f'  ✓ train: {train_chunk.shape} ({train_chunk.nbytes/1e9:.2f} GB)')
    del train_chunk
    val_chunk = np.array(arr[n_train:n_train+n_val_rows])
    val_flat = val_chunk.reshape(-1, val_chunk.shape[-1])
    np.save(VAL_DST, val_flat)
    print(f'  ✓ val: {val_flat.shape} ({val_flat.nbytes/1e6:.0f} MB)')
    del val_chunk, val_flat
    os.remove(SRC)
    print('  removed 17 GB original')

## 10. Config = **bayan**


In [ ]:
import sys
sys.path.insert(0, '/content')
from bayan_helpers import ensure_bayan_config
cfg = ensure_bayan_config(force=True)
print('target_phrase:', cfg['target_phrase'])
print('model_name:   ', cfg['model_name'])
print('output_dir:   ', cfg['output_dir'])


## 11. Generate TTS clips for bayan


In [ ]:
import os, sys, uuid
from pathlib import Path
import torch

OWW_DIR, PSG_DIR = '/content/openwakeword', '/content/piper-sample-generator'
sys.path[:0] = [OWW_DIR, PSG_DIR, '/content']
os.environ['PYTHONPATH'] = OWW_DIR
if not getattr(torch.load, '_bayan_patched', False):
    _orig = torch.load
    def _load(*a, **k):
        k.setdefault('weights_only', False)
        return _orig(*a, **k)
    _load._bayan_patched = True
    torch.load = _load

from bayan_helpers import ensure_bayan_config
cfg = ensure_bayan_config()
phrases = cfg['target_phrase']
out_root = Path(cfg['output_dir']) / cfg['model_name']
dirs = {k: out_root / k for k in ['positive_train','positive_test','negative_train','negative_test']}
for p in dirs.values():
    p.mkdir(parents=True, exist_ok=True)

n_train, n_val = int(cfg['n_samples']), int(cfg['n_samples_val'])
batch = min(25, int(cfg.get('tts_batch_size', 50)))

def count(p): return len(list(Path(p).glob('*.wav')))
def need(p, t): return count(p) < int(t * 0.75)

print('Model', cfg['model_name'], phrases)
from generate_samples import generate_samples

SIMPLE = ['banana','bay','yon','hey man','buy one','by and','brian','diane','zion',
          'okay then','hey buddy','beyond','begin','hey there','day one','pay one',
          'they can','hi there','bayon','mayan','hayan','beige','high bay']

def gen(texts, folder, n, batch_size=20):
    todo = max(0, n - count(folder))
    if todo <= 0:
        print('skip', folder.name, count(folder)); return
    print('gen', todo, '->', folder.name)
    generate_samples(
        text=texts, max_samples=todo, batch_size=batch_size,
        noise_scales=[0.98], noise_scale_ws=[0.98], length_scales=[0.75,1.0,1.25],
        output_dir=str(folder), auto_reduce_batch_size=True,
        file_names=[uuid.uuid4().hex + '.wav' for _ in range(todo)],
    )
    print('now', count(folder), folder.name)

if need(dirs['positive_train'], n_train): gen(phrases, dirs['positive_train'], n_train, batch)
else: print('skip positive_train')
if need(dirs['positive_test'], n_val): gen(phrases, dirs['positive_test'], n_val, batch)
else: print('skip positive_test')

neg = list(SIMPLE)
while len(neg) < n_train: neg += SIMPLE
nb_ = max(1, batch // 7)
if need(dirs['negative_train'], n_train): gen(neg, dirs['negative_train'], n_train, nb_)
else: print('skip negative_train')
if need(dirs['negative_test'], n_val): gen(neg, dirs['negative_test'], n_val, nb_)
else: print('skip negative_test')

ok = True
for name, t in [('positive_train', n_train), ('positive_test', n_val),
                ('negative_train', n_train), ('negative_test', n_val)]:
    n = count(dirs[name]); exp = int(t * 0.75)
    print(('PASS' if n >= exp else 'MISS'), name, n, '>=', exp)
    ok &= n >= exp
assert ok, 'clips incomplete'
print('PASS clips ready')


## 12. Resample to 16 kHz


In [ ]:
import os, glob, sys, soundfile as sf
from scipy.signal import resample_poly
from tqdm.auto import tqdm
sys.path.insert(0, '/content')
from bayan_helpers import ensure_bayan_config
cfg = ensure_bayan_config()
TARGET_SR, ROOT = 16000, cfg['output_dir']
wav_dirs = sorted({os.path.dirname(f) for f in glob.glob(f'{ROOT}/**/*.wav', recursive=True)})
assert wav_dirs, 'no wavs — re-run cell 11'
for d in wav_dirs:
    files = [f for f in os.listdir(d) if f.endswith('.wav')]
    if not files: continue
    sample = sf.info(f'{d}/{files[0]}').samplerate
    if sample == TARGET_SR:
        print('ok', d, len(files)); continue
    print('resample', d, len(files), 'from', sample)
    for f in tqdm(files, leave=False):
        p = f'{d}/{f}'; data, sr = sf.read(p)
        if sr != TARGET_SR:
            sf.write(p, resample_poly(data.astype('float32'), TARGET_SR, sr), TARGET_SR)
for f in glob.glob(f'{ROOT}/**/*.npy', recursive=True):
    os.remove(f)
print('cleared stale npy features')


## 13. Augment + featurise


In [ ]:
import sys, os, glob, subprocess
sys.path.insert(0, '/content')
from bayan_helpers import ensure_bayan_config
cfg = ensure_bayan_config()
FEAT = cfg['feature_save_dir']
os.makedirs(FEAT, exist_ok=True)
os.makedirs('/content/mit_rirs', exist_ok=True)
os.makedirs('/content/fma_wav', exist_ok=True)
needed = ['positive_features_train.npy','negative_features_train.npy',
          'positive_features_test.npy','negative_features_test.npy']
if all(os.path.exists(f'{FEAT}/{n}') for n in needed):
    print('cached features OK')
else:
    env = {**os.environ, 'PYTHONPATH': '/content/openwakeword'}
    print('running augment_clips...')
    p = subprocess.run(
        [sys.executable, '-m', 'openwakeword.train',
         '--training_config', '/content/my_model.yaml', '--augment_clips'],
        env=env, cwd='/content', capture_output=True, text=True)
    if p.stdout: print(p.stdout[-5000:])
    if p.returncode != 0:
        if p.stderr: print(p.stderr[-5000:])
        raise RuntimeError(f'augment failed rc={p.returncode}')
for f in sorted(glob.glob(f'{FEAT}/*.npy')):
    print(f, f'{os.path.getsize(f)/1e6:.1f} MB')
for n in needed:
    assert os.path.exists(f'{FEAT}/{n}'), n
print('PASS features')


## 14. Train model (~30–40 min)


In [ ]:
import os, copy, math, time
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import yaml
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
import sys as _sys
_sys.path.insert(0, '/content')
from bayan_helpers import ensure_bayan_config
cfg = ensure_bayan_config()
FEAT = cfg['feature_save_dir']
pos_train = torch.from_numpy(np.load(f'{FEAT}/positive_features_train.npy').astype(np.float32)).to(DEVICE)
neg_train = torch.from_numpy(np.load(f'{FEAT}/negative_features_train.npy').astype(np.float32)).to(DEVICE)
pos_test  = torch.from_numpy(np.load(f'{FEAT}/positive_features_test.npy').astype(np.float32)).to(DEVICE)
neg_test  = torch.from_numpy(np.load(f'{FEAT}/negative_features_test.npy').astype(np.float32)).to(DEVICE)
print(f'  pos_train={tuple(pos_train.shape)}  neg_train={tuple(neg_train.shape)}')
print(f'  pos_test={tuple(pos_test.shape)}  neg_test={tuple(neg_test.shape)}')

acav_train_np = np.load(cfg['feature_data_files']['ACAV100M_sample'], mmap_mode='r')
print(f'  acav_train (mmap): {acav_train_np.shape}')
acav_val_np = np.load(cfg['false_positive_validation_data_path'])
M = acav_val_np.shape[0]
val_listen_hours = M * 0.08 / 3600.0
print(f'  acav_val: {acav_val_np.shape}  → {val_listen_hours:.2f} hr listening time')
n_win_val = M - 16
acav_val_windows = np.lib.stride_tricks.sliding_window_view(acav_val_np, (16, 96))[:, 0, :, :]
acav_val_windows = np.ascontiguousarray(acav_val_windows.astype(np.float32))
print(f'  acav_val_windows: {acav_val_windows.shape} ({acav_val_windows.nbytes/1e6:.0f} MB)')
VAL_BATCH = 4096

class WakewordModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.layer1 = nn.Linear(16 * 96, 128)
        self.layernorm1 = nn.LayerNorm(128)
        self.relu1 = nn.ReLU()
        self.layer2 = nn.Linear(128, 1)
    def forward(self, x):
        return self.layer2(self.relu1(self.layernorm1(self.layer1(self.flatten(x)))))

model = WakewordModel().to(DEVICE)
loss_fn = nn.BCEWithLogitsLoss(reduction='none')
TOTAL_STEPS = cfg['steps']
MAX_NEG_W   = cfg['max_negative_weight']
TARGET_FP_PER_HR = cfg['target_false_positives_per_hour']
THRESH = 0.5
history = {'val_recall': [], 'val_accuracy': [], 'val_fp_per_hour': [], 'val_n_fp': [], 'loss': []}
best_models = []

B_POS, B_ANEG, B_ACAV = 32, 32, 64
def random_acav_window_batch(k):
    N, T, F = acav_train_np.shape
    rows = np.random.randint(0, N, size=k)
    starts = np.random.randint(0, T - 16 + 1, size=k)
    out = np.empty((k, 16, F), dtype=np.float32)
    for i, (r, s) in enumerate(zip(rows, starts)):
        out[i] = acav_train_np[r, s:s+16, :].astype(np.float32)
    return out

def build_batch():
    p_idx = torch.randint(0, pos_train.shape[0], (B_POS,), device=DEVICE)
    aneg_idx = torch.randint(0, neg_train.shape[0], (B_ANEG,), device=DEVICE)
    p, an = pos_train[p_idx], neg_train[aneg_idx]
    acav = torch.from_numpy(random_acav_window_batch(B_ACAV)).to(DEVICE)
    x = torch.cat([p, an, acav], dim=0)
    y = torch.cat([torch.ones(B_POS, device=DEVICE),
                   torch.zeros(B_ANEG + B_ACAV, device=DEVICE)])
    return x, y

@torch.no_grad()
def validate(step_label):
    model.eval()
    p_preds = torch.sigmoid(model(pos_test)).squeeze(-1)
    n_preds = torch.sigmoid(model(neg_test)).squeeze(-1)
    recall = (p_preds >= THRESH).float().mean().item()
    accuracy = (((p_preds >= THRESH).sum() + (n_preds < THRESH).sum()).item()
                / (pos_test.shape[0] + neg_test.shape[0]))
    n_fp = 0
    for i in range(0, n_win_val, VAL_BATCH):
        chunk = torch.from_numpy(acav_val_windows[i:i+VAL_BATCH]).to(DEVICE)
        n_fp += (torch.sigmoid(model(chunk)).squeeze(-1) >= THRESH).sum().item()
    fp_per_hour = n_fp / max(val_listen_hours, 1e-6)
    history['val_recall'].append(recall)
    history['val_accuracy'].append(accuracy)
    history['val_fp_per_hour'].append(fp_per_hour)
    history['val_n_fp'].append(n_fp)
    save = False
    if len(history['val_n_fp']) >= 3:
        fp_p50 = np.percentile(history['val_n_fp'], 50)
        rc_p5  = np.percentile(history['val_recall'], 5)
        if n_fp <= fp_p50 and recall >= rc_p5:
            best_models.append((copy.deepcopy(model.state_dict()),
                                {'val_recall': recall, 'val_accuracy': accuracy,
                                 'val_fp_per_hour': fp_per_hour, 'val_n_fp': n_fp}))
            save = True
    print(f'  [{step_label}] recall={recall:.3f}  acc={accuracy:.3f}  fp/hr={fp_per_hour:.2f}  saved={"+" if save else "-"}', flush=True)
    model.train()

def run_stage(stage_idx, n_steps, lr, max_neg_w, val_window_frac=1.0):
    optimizer = optim.Adam(model.parameters(), lr=lr)
    weight_schedule = np.linspace(1.0, max_neg_w, n_steps)
    val_start = int(n_steps * (1.0 - val_window_frac))
    val_steps = set(np.linspace(val_start, n_steps - 1, 20).astype(int))
    warmup = max(1, n_steps // 5)
    hold   = n_steps // 3
    accumulated = []
    print(f'\n=== Stage {stage_idx}: {n_steps} steps, lr={lr}, max_neg_w={max_neg_w} ===', flush=True)
    t0 = time.time()
    for step in range(n_steps):
        if step < warmup:
            lr_now = lr * (step + 1) / warmup
        elif step < warmup + hold:
            lr_now = lr
        else:
            decay_t = (step - warmup - hold) / max(1, n_steps - warmup - hold)
            lr_now = lr * 0.5 * (1.0 + math.cos(math.pi * min(1.0, decay_t)))
        for pg in optimizer.param_groups: pg['lr'] = lr_now
        x, y = build_batch()
        logits = model(x).squeeze(-1)
        preds = torch.sigmoid(logits)
        keep = ((y == 0) & (preds >= 0.001)) | ((y == 1) & (preds < 0.999))
        if keep.sum() == 0:
            if step in val_steps: validate(f'stage{stage_idx} step {step}/{n_steps}')
            continue
        kept_logits = logits[keep]
        kept_y = y[keep]
        neg_w = weight_schedule[step]
        w = torch.where(kept_y > 0.5,
                        torch.tensor(1.0, device=DEVICE),
                        torch.tensor(neg_w, device=DEVICE, dtype=torch.float32))
        accumulated.append((kept_logits, kept_y, w))
        if sum(t[0].shape[0] for t in accumulated) >= 128:
            cat_logits = torch.cat([t[0] for t in accumulated])
            cat_y = torch.cat([t[1] for t in accumulated])
            cat_w = torch.cat([t[2] for t in accumulated])
            loss = (loss_fn(cat_logits, cat_y) * cat_w).mean()
            optimizer.zero_grad(); loss.backward(); optimizer.step()
            history['loss'].append(loss.item())
            accumulated.clear()
        if step in val_steps:
            elapsed = (time.time() - t0) / 60
            validate(f'stage{stage_idx} step {step}/{n_steps} ({elapsed:.1f}m, lr={lr_now:.6f}, neg_w={neg_w:.0f})')
    print(f'  Stage {stage_idx} done in {(time.time()-t0)/60:.1f} min', flush=True)

stage1_steps = TOTAL_STEPS
stage2_steps = max(2000, TOTAL_STEPS // 10)
stage3_steps = max(2000, TOTAL_STEPS // 10)
max_neg_w_now = MAX_NEG_W
run_stage(1, stage1_steps, lr=1e-4, max_neg_w=max_neg_w_now, val_window_frac=0.25)
if (history['val_fp_per_hour'] and min(history['val_fp_per_hour']) > TARGET_FP_PER_HR):
    max_neg_w_now *= 2
    print(f'\n[adapt] stage1 best FP/hr > target → max_neg_w doubled to {max_neg_w_now}')
run_stage(2, stage2_steps, lr=1e-5, max_neg_w=max_neg_w_now, val_window_frac=1.0)
if (history['val_fp_per_hour'] and min(history['val_fp_per_hour']) > TARGET_FP_PER_HR):
    max_neg_w_now *= 2
    print(f'\n[adapt] stage2 best FP/hr > target → max_neg_w doubled to {max_neg_w_now}')
run_stage(3, stage3_steps, lr=1e-6, max_neg_w=max_neg_w_now, val_window_frac=1.0)
print(f'\n=== Training done. {len(best_models)} checkpoints saved. ===')
print(f'  best val_fp_per_hour: {min(history["val_fp_per_hour"]):.2f}')
print(f'  best val_recall:      {max(history["val_recall"]):.3f}')
print(f'  best val_accuracy:    {max(history["val_accuracy"]):.3f}')


## 15. Export **bayan.onnx** + download


In [ ]:
import copy, os
import numpy as np
import torch
if not best_models:
    print('  no saved checkpoints — using current model state')
    final_state = {k: v.clone() for k, v in model.state_dict().items()}
else:
    accs = [s['val_accuracy'] for _, s in best_models]
    rcs  = [s['val_recall']   for _, s in best_models]
    fps  = [s['val_fp_per_hour'] for _, s in best_models]
    acc_p90 = np.percentile(accs, 90)
    rc_p90  = np.percentile(rcs, 90)
    fp_p10  = np.percentile(fps, 10)
    print(f'  thresholds: acc≥{acc_p90:.3f}  recall≥{rc_p90:.3f}  fp/hr≤{fp_p10:.2f}')
    qualified = [(sd, sc) for sd, sc in best_models
                 if sc['val_accuracy'] >= acc_p90 and sc['val_recall'] >= rc_p90
                 and sc['val_fp_per_hour'] <= fp_p10]
    print(f'  qualified: {len(qualified)} of {len(best_models)} checkpoints')
    if not qualified:
        sorted_models = sorted(best_models, key=lambda t: (t[1]['val_fp_per_hour'], -t[1]['val_recall']))
        qualified = [sorted_models[0]]
        print('  no checkpoint qualified — fell back to single best')
    keys = qualified[0][0].keys()
    final_state = {k: torch.stack([sd[k].float() for sd, _ in qualified]).mean(dim=0) for k in keys}
    qa = np.mean([s['val_accuracy'] for _, s in qualified])
    qr = np.mean([s['val_recall'] for _, s in qualified])
    qf = np.mean([s['val_fp_per_hour'] for _, s in qualified])
    print(f'  ensemble averaged: mean acc={qa:.3f}, recall={qr:.3f}, fp/hr={qf:.2f}')
model.load_state_dict(final_state)
model.eval()

# Export with sigmoid baked in so APK runtime sees [0, 1]
out_path = f"{cfg['feature_save_dir']}/{cfg['model_name']}.onnx"
os.makedirs(os.path.dirname(out_path), exist_ok=True)
class WakewordExportable(torch.nn.Module):
    def __init__(self, base): super().__init__(); self.base = base
    def forward(self, x): return torch.sigmoid(self.base(x))
export_model = WakewordExportable(model).to(DEVICE).eval()
dummy = torch.randn(1, 16, 96, device=DEVICE)
torch.onnx.export(export_model, dummy, out_path,
                  input_names=['onnx::Flatten_0'], output_names=['output'],
                  dynamic_axes={'onnx::Flatten_0': {0: 'batch'}, 'output': {0: 'batch'}},
                  opset_version=14, dynamo=False)
print(f'  ✓ wrote {out_path} ({os.path.getsize(out_path)/1e3:.0f} KB)')

import onnxruntime as ort
sess = ort.InferenceSession(out_path)
print(f'  input:  {sess.get_inputs()[0].name} {sess.get_inputs()[0].shape}')
print(f'  output: {sess.get_outputs()[0].name} {sess.get_outputs()[0].shape}')
with torch.no_grad():
    p_test_np = pos_test.cpu().numpy()
    p_scores = sess.run(None, {sess.get_inputs()[0].name: p_test_np})[0].flatten()
    print(f'  positive test set: mean={p_scores.mean():.3f}, recall@0.5={(p_scores>=0.5).mean():.3f}')
print(f'  ACAV val FP/hour at 0.5: {history["val_fp_per_hour"][-1]:.2f}')

from google.colab import files
files.download(out_path)
print(f'\nDONE. Wake word "{cfg["model_name"]}" trained.')
print('Drop the downloaded .onnx into your APK / on-device app at:')
print(f'  android/app/src/main/assets/wakewords/{cfg["model_name"]}.onnx  (or wherever your APK looks)')
